In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in CANDIDATES if (path / "data" / "processed" / "sdm").exists())
SDM_DIR = REPO_ROOT / "data" / "processed" / "sdm"
confusion_matrix = SDM_DIR / "confusion_summary.csv"

df = pd.read_csv(confusion_matrix)

In [ ]:
# Choose metric to plot: 'count_records' or 'count_weighted'
metric_col = 'count_records'

# Build confusion matrix table
cm = (
    df.pivot(index='actual_label', columns='predicted_label', values=metric_col)
      .fillna(0)
      .sort_index()
      .reindex(sorted(df['predicted_label'].unique()), axis=1)
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm.values, cmap='Blues')

# Axis labels and ticks
ax.set_xticks(np.arange(cm.shape[1]))
ax.set_yticks(np.arange(cm.shape[0]))
ax.set_xticklabels(cm.columns)
ax.set_yticklabels(cm.index)
ax.set_xlabel('Predicted label')
ax.set_ylabel('Actual label')

# Annotate cell values
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm.iloc[i, j]
        txt = f'{int(v):,}' if float(v).is_integer() else f'{v:,.2f}'
        ax.text(j, i, txt, ha='center', va='center', color='black')

fig.colorbar(im, ax=ax, label=metric_col)
plt.tight_layout()
plt.show()


In [ ]:
# ROC/AUC from per-sample predictions
from sklearn.metrics import roc_curve, roc_auc_score

predictions_path = SDM_DIR / 'capture_predictions.csv'
pred_df = pd.read_csv(predictions_path)

y_true = pred_df['actual_presence'].astype(int)
y_score = pred_df['probability'].astype(float)

fpr, tpr, _ = roc_curve(y_true, y_score)
auc_value = roc_auc_score(y_true, y_score)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {auc_value:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Random baseline')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print(f'AUC: {auc_value:.6f}')
